# Notebook 01: Data Pipeline

Downloads and processes raw data. Produces abagym_antibody.csv, abagym_sequences.csv, sabdab_affinity.csv on Drive. This notebook was already run successfully. It exists for documentation and reproducibility -- re-run to verify outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys

REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
REPO_DIR = '/content/antibody-property-prediction'
BRANCH = 'implementation'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project')
# DATA_DIR is in the repo (data/ at repo root) -- comes from src.config
EMBEDDING_DIR = DRIVE_ROOT / 'embeddings'
RESULTS_DIR = DRIVE_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set.")

In [ ]:
!apt-get install -y hmmer
!pip install -q fair-esm ablang2 anarci wandb

In [ ]:
!pip install -q --upgrade ipython

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
subprocess.run(['find', '/content/antibody-property-prediction', '-type', 'd',
                '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
               capture_output=True)
print("Autoreload enabled, pycache cleared.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import wandb
wandb.login()

## Imports

In [ ]:
from src.data.abagym import load_abagym_antibody, load_abagym_sequences
from src.data.sabdab import load_sabdab
from src.config import DATA_DIR, ABAGYM_DATASETS, N_MUTATIONS, N_SABDAB

## AbAgym Download

Download DMS datasets from github.com/3BioCompBio/AbAgym. One CSV per antibody.

## SAbDab Download

Download binding affinity data from Zenodo DOI 10.5281/zenodo.13120765 (Apache 2.0). Do NOT use PyTDC -- incompatible with Colab numpy.

## Mutant Sequence Reconstruction

Reconstruct full mutant sequences from wildtype + single-point substitution. Store in mutant_heavy_seq and mutant_light_seq columns.

## CDR/FR Mapping

Map PDB position labels to IMGT positions using ANARCI. Requires HMMER (installed in cell 3). Pipeline: PDB position -> ANARCI alignment -> IMGT position -> CDR/FR region label.

Verification: 4zfg chain H, PDB 100A -> IMGT 113 -> CDR_H3.

## Verification

Assert expected row counts and column presence before saving.

In [ ]:
# antibody_df = load_abagym_antibody(DATA_DIR)
# sequences_df = load_abagym_sequences(DATA_DIR)
# sabdab_df = load_sabdab(DATA_DIR)
# print(antibody_df.shape, sequences_df.shape, sabdab_df.shape)

## Save to Drive